In [1]:
# importar la api key y el tenant
import os
import pandas as pd
import json
import requests

CONFIG_PATH = os.path.join("..","..","config.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

c:\Users\dblan\anaconda3\envs\Assistants_con\lib\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [9]:
TENANT = config["Qlik_connection"]["Qlik_tenant"]
API_KEY= config["Qlik_connection"]["Qlik_4s"]

base = f"https://{TENANT}/api/v1"
headers = {"Authorization": f"Bearer {API_KEY}"}

In [10]:
resp = requests.get(f"{base}/apps", headers=headers)
print("Status code:", resp.status_code)
print("Response body:", resp.text)

Status code: 200
Response body: {"data":[{"attributes":{"id":"0d83ade6-9d52-4da1-8448-59341f421d08","name":"REDI LATAM","description":"","thumbnail":"/api/v1/apps/0d83ade6-9d52-4da1-8448-59341f421d08/media/files/1Monterrey.png","lastReloadTime":"2025-07-19T10:56:16.258Z","createdDate":"2025-02-06T19:05:30.048Z","modifiedDate":"2025-07-19T10:56:39.854Z","owner":"auth0|47f8a54afdebc603e7753184154e0aeaedfc1d3a970962d1355e2398671868ad","ownerId":"677dc3b540acfc79eec6dac3","dynamicColor":"","published":false,"publishTime":"","custom":{},"hasSectionAccess":true,"encrypted":true,"originAppId":"","isDirectQueryMode":false,"usage":"ANALYTICS","spaceId":"67a4fa985f80476eb2be74b5","_resourcetype":"app"},"privileges":[],"create":[]},{"attributes":{"id":"0e57ff25-9f70-4973-8134-78add8ada5a6","name":"4S Interno - Definición","description":"","thumbnail":"","lastReloadTime":"2025-07-19T07:11:59.817Z","createdDate":"2025-02-23T03:15:57.437Z","modifiedDate":"2025-07-19T07:12:25.775Z","owner":"auth0|47f

In [11]:
apps = resp.json().get("data", [])
print("Apps encontradas:", [a["attributes"]["name"] for a in apps])
print("IDs:", [a["attributes"]["id"] for a in apps])

Apps encontradas: ['REDI LATAM', '4S Interno - Definición', 'App Analyzer', '4S Interno - Auditoría', 'Reload Analyzer', '4S Interno - Auditoria Big Data', 'DEV - Investigación', '4S Corporativo', 'REDI - Testing Stage', 'Access Evaluator', '4S Interno - Gran Reporte de Verticalización', '4S Demo ELDI', 'REDI Database Extraction', 'REDI Mx - backup', 'REDI - Retail', 'Report Analyzer', 'TECH - Widgets', 'Answers Analyzer', 'Consumption report 2025-03-21', 'TECH - Camilo B.', 'test_core_data', 'REDI - Real Estate Data Insights', 'TECH Real Estate Data Insights - OV', 'REDI - Pruebas Tech', 'Script - 4S Interno - Inf Secundaria', '4S Interno - Opinion de Valor', 'Info Secundaria Loader', 'REDI Financiero', 'AI_chatbot_test', '4S Interno - Test Relacion Geografica inf secundaria', '4S Interno - Inf Sec (no usar)', 'Automation Analyzer', 'Test', 'Access Evaluator_Tenant ant.', '4S Interno - Inf Secundaria', 'TECH - Estudio Vertical (Pruebas)', 'TECH - REDI Interno', 'Consumption report 202

In [16]:
import websocket, json

# Variables de conexión
APP_ID = config["Qlik_connection"]["AI_chatbot_test"]
url = f"wss://{TENANT}/app/{APP_ID}"

def on_message(ws, msg):
    resp = json.loads(msg)
    print(f"DEBUG – ID recibido: {resp.get('id')}")

    # 1) Tras OpenDoc (id=1), pedir el objeto de tabla existente
    if resp.get("id") == 1:
        doc_handle = resp["result"]["qReturn"]["qHandle"]
        ws.send(json.dumps({
            "jsonrpc":"2.0","id":2,"handle":doc_handle,
            "method":"GetObject",
            "params":{"qId":"BCSxDL"}  # <-- tu qId de tabla
        }))

    # 2) Obtener handle del objeto de tabla
    elif resp.get("id") == 2:
        tbl_handle = resp["result"]["qReturn"]["qHandle"]
        ws.send(json.dumps({
            "jsonrpc":"2.0","id":3,"handle":tbl_handle,
            "method":"GetLayout","params":{}
        }))

    # 3) Leer tamaño
    elif resp.get("id") == 3:
        size = resp["result"]["qLayout"]["qHyperCube"]["qSize"]
        pages = [{
            "qTop": 0,
            "qLeft": 0,
            "qHeight": min(1000, size["qcy"]),
            "qWidth": size["qcx"]
        }]
        # DEBUG: mostrar payload antes de enviar
        payload = {
            "jsonrpc": "2.0",
            "id": 4,
            "handle": table_handle,       # usar table_handle, no resp["handle"]
            "method": "GetHyperCubeData",
            "params": ["/qHyperCubeDef", pages]
        }
        print(f"DEBUG – Enviando GetHyperCubeData: {payload}")
        ws.send(json.dumps(payload))

    # 4) Mostrar filas y cerrar conexión
    elif resp.get("id") == 4:
        matrix = resp["result"]["qDataPages"][0]["qMatrix"]
        print("Filas de la tabla:")
        for row in matrix:
            print([c.get("qText","") for c in row])
        ws.close()   # <-- cierra la conexión y termina run_forever()

def on_open(ws):
    ws.send(json.dumps({
        "jsonrpc":"2.0","id":1,"handle":-1,
        "method":"OpenDoc","params":{"qDocName":APP_ID}
    }))

ws = websocket.WebSocketApp(
    url, on_message=on_message, on_open=on_open,
    header=[f"Authorization: Bearer {API_KEY}", "Sec-WebSocket-Protocol: qlik.api"]
)
ws.run_forever(sslopt={"cert_reqs":0})


DEBUG – ID recibido: None
DEBUG – ID recibido: 1
DEBUG – ID recibido: 2
DEBUG – ID recibido: 3
DEBUG – Enviando GetHyperCubeData: {'jsonrpc': '2.0', 'id': 4, 'handle': 2, 'method': 'GetHyperCubeData', 'params': ['/qHyperCubeDef', [{'qTop': 0, 'qLeft': 0, 'qHeight': 1000, 'qWidth': 9}]]}
DEBUG – ID recibido: 4
Filas de la tabla:
['5311', '132215', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132216', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132217', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132218', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132219', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132220', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132221', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132222', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132223', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '132224', '1', '2', '3 Recámaras', '0', '0', '2', '134']
['5311', '

False

## algo pasa con table handel que no permite mostrar los datos de la tabla, no le he pasado este resultado a GPT par tratar de solucionarlo.